## Usefull libraries

In [2]:

import numpy as np
from scipy.stats import chi2
import glob
import camb
from camb import model, initialpower
import math
import matplotlib.pyplot as plt
from scipy import special
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord , CartesianRepresentation
from scipy.integrate import quad
from scipy.interpolate import interp1d
from scipy.integrate import simpson
from scipy.integrate import trapezoid
from CosmoFunc import *
from BF_OptMC import *
import time
import os
from numba import jit, prange

## Calculation of the CRMS of LCDM prediction

In [6]:
# extraction of all the file of N galaxies of the R matrix : 
N = 3000 # change here to the number of galaxies you have
files = sorted(glob.glob(f'/renoir/fromenti/Documents/codes_Bulk_flow/random/R_matrix_l_b_{N}g_sample_*.txt')) # cha,ge here to your file name pattern

# extraction of the R matrix from the files
R = []
for fichier in files:
    mat = np.loadtxt(fichier, skiprows=2)
    R.append(mat)

# store each R matrix in a separate variable (R1, R2, ..., R10)
for i, mat in enumerate(R, 1):
    globals()[f'R{i}'] = mat
    
# calculation of the mean R matrix across all matrices
R_mean = np.mean(R, axis=0)

# calculation of the standard deviation of the R matrix across all matrices
R_std = np.std(R, axis=0)

# print the mean R matrix : 
for row in R_mean:
    print("  ".join(f"{x:10.4f}" for x in row))



18369.1558  -14395.1996  -10686.9512     27.1859    -43.2621    -37.4430     38.0571     43.0881     15.7978
-14395.1996  25423.3460  14732.0000    -16.1686     49.4334     40.2544    -69.2271    -63.3759    -25.1834
-10686.9512  14732.0000  16073.8335    -13.2087     33.4167     32.9149    -40.4232    -49.8325    -26.7516
   27.1859    -16.1686    -13.2087      0.2257     -0.0778     -0.0465      0.0842      0.0396      0.0513
  -43.2621     49.4334     33.4167     -0.0778      0.1860      0.1100     -0.1459     -0.1371     -0.0499
  -37.4430     40.2544     32.9149     -0.0465      0.1100      0.1420     -0.1003     -0.1338     -0.0632
   38.0571    -69.2271    -40.4232      0.0842     -0.1459     -0.1003      0.3698      0.1807      0.1048
   43.0881    -63.3759    -49.8325      0.0396     -0.1371     -0.1338      0.1807      0.2386      0.1002
   15.7978    -25.1834    -26.7516      0.0513     -0.0499     -0.0632      0.1048      0.1002      0.1968


In [35]:
# Bulk Flow CRMS
B_pq = R_mean[:3, :3]
Bx_LCDM = np.sqrt(B_pq[0, 0])
By_LCDM = np.sqrt(B_pq[1, 1])
Bz_LCDM = np.sqrt(B_pq[2, 2])
B_LCDM = np.sqrt(np.trace(B_pq))
    
# Shear moments CRMS
R_shear = R_mean[3:8, 3:8]
Qxx_LCDM= np.sqrt(R_shear[0, 0])
Qxy_LCDM = np.sqrt(R_shear[1, 1])
Qxz_LCDM = np.sqrt(R_shear[2, 2])
Qyy_LCDM = np.sqrt(R_shear[3, 3])
Qyz_LCDM = np.sqrt(R_shear[4, 4])
Qzz_LCDM = np.sqrt(np.abs(-R_shear[0, 0] - R_shear[3, 3]))

print('The theoritical Bulk Flow Bx = 0 ± ',Bx_LCDM,'km/s')
print('The theoritical Bulk Flow By = 0 ± ',By_LCDM,'km/s')
print('The theoritical Bulk Flow Bz = 0 ± ',Bz_LCDM,'km/s')
print('The theoritical Bulk Flow amplitude |B| =' , 0, '± ',B_LCDM,'km/s')

print("Prédiction ΛCDM pour les shear moments :")
print(f"Qxx = 0 ± {Qxx_LCDM:.4f} km/s")
print(f"Qyy = 0 ± {Qyy_LCDM:.4f} km/s")
print(f"Qxy = 0 ± {Qxy_LCDM:.4f} km/s")
print(f"Qxz = 0 ± {Qxz_LCDM:.4f} km/s")
print(f"Qyy = 0 ± {Qyy_LCDM:.4f} km/s")
print(f"Qyz = 0 ± {Qyz_LCDM:.4f} km/s")
print(f"Qzz = 0 ± {Qzz_LCDM:.4f} km/s")

The theoritical Bulk Flow Bx = 0 ±  135.53285893096182 km/s
The theoritical Bulk Flow By = 0 ±  159.44700070347304 km/s
The theoritical Bulk Flow Bz = 0 ±  126.78262295493549 km/s
The theoritical Bulk Flow amplitude |B| = 0 ±  244.67598036314612 km/s
Prédiction ΛCDM pour les shear moments :
Qxx = 0 ± 0.4751 km/s
Qyy = 0 ± 0.6081 km/s
Qxy = 0 ± 0.4313 km/s
Qxz = 0 ± 0.3768 km/s
Qyy = 0 ± 0.6081 km/s
Qyz = 0 ± 0.4884 km/s
Qzz = 0 ± 0.7717 km/s


## Extraction of the moments for the data and the mocks 

In [28]:
import numpy as np

def load_data_all(filepath_data, filepath_mocks):
    with open(filepath_data) as f:
        l = f.readlines()
    
    def g(i, j): return float(l[i].split()[j])
    
    # Bulk Flow
    B_ampl, err_B_ampl = g(8,1), g(8,2)
    Bx, err_Bx = g(9,1), g(9,2)
    By, err_By = g(10,1), g(10,2)
    Bz, err_Bz = g(11,1), g(11,2)
    
    # Shear moments
    Qxx, err_Qxx = g(17,1), g(17,2)
    Qxy, err_Qxy = g(18,1), g(18,2)
    Qxz, err_Qxz = g(19,1), g(19,2)
    Qyy, err_Qyy = g(20,1), g(20,2)
    Qyz, err_Qyz = g(21,1), g(21,2)
    Qzz, err_Qzz = g(22,1), g(22,2)
    
    # SigV and covariance
    SigV = g(24,4)
    cov = np.array([[float(x) for x in l[28+i].split()] for i in range(8)])
    
    # Mocks
    m = np.loadtxt(filepath_mocks, skiprows=1)
    
    # Mean of the mocks 
    Bx_opt_mocks_mean, By_opt_mocks_mean, Bz_opt_mocks_mean = np.mean(m[:,13:16], axis=0)
    Bx_true_mocks_mean, By_true_mocks_mean, Bz_true_mocks_mean = np.mean(m[:,16:19], axis=0)
    Qxx_opt_mocks_mean, Qxy_opt_mocks_mean, Qxz_opt_mocks_mean, Qyy_opt_mocks_mean, Qyz_opt_mocks_mean, Qzz_opt_mocks_mean = np.mean(m[:,1:7], axis=0)
    Qxx_true_mocks_mean, Qxy_true_mocks_mean, Qxz_true_mocks_mean, Qyy_true_mocks_mean, Qyz_true_mocks_mean, Qzz_true_mocks_mean = np.mean(m[:,7:13], axis=0)
    
    # Standard deviations of the mocks
    Bx_opt_mocks_err, By_opt_mocks_err, Bz_opt_mocks_err = np.std(m[:,13:16], axis=0, ddof=1)
    Bx_true_mocks_err, By_true_mocks_err, Bz_true_mocks_err = np.std(m[:,16:19], axis=0, ddof=1)
    Qxx_opt_mocks_CRMS, Qxy_opt_mocks_CRMS, Qxz_opt_mocks_CRMS, Qyy_opt_mocks_CRMS, Qyz_opt_mocks_CRMS, Qzz_opt_mocks_CRMS = np.std(m[:,1:7], axis=0, ddof=1)
    Qxx_true_mocks_CRMS, Qxy_true_mocks_CRMS, Qxz_true_mocks_CRMS, Qyy_true_mocks_CRMS, Qyz_true_mocks_CRMS, Qzz_true_mocks_CRMS = np.std(m[:,7:13], axis=0, ddof=1)
    
    # Magnitudes
    B_opt_mocks_mean = np.sqrt(Bx_opt_mocks_mean**2 + By_opt_mocks_mean**2 + Bz_opt_mocks_mean**2)
    B_true_mocks_mean = np.sqrt(Bx_true_mocks_mean**2 + By_true_mocks_mean**2 + Bz_true_mocks_mean**2)
    B_opt_mocks_err = np.sqrt(Bx_opt_mocks_err**2 + By_opt_mocks_err**2 + Bz_opt_mocks_err**2)
    B_true_mocks_err = np.sqrt(Bx_true_mocks_err**2 + By_true_mocks_err**2 + Bz_true_mocks_err**2)

    return (Bx, By, Bz, B_ampl, err_Bx, err_By, err_Bz, err_B_ampl,
            Qxx, Qxy, Qxz, Qyy, Qyz, Qzz,
            err_Qxx, err_Qxy, err_Qxz, err_Qyy, err_Qyz, err_Qzz,
            cov, SigV,
            Bx_opt_mocks_err, By_opt_mocks_err, Bz_opt_mocks_err,
            Bx_true_mocks_err, By_true_mocks_err, Bz_true_mocks_err,
            B_opt_mocks_err, B_true_mocks_err, B_opt_mocks_mean, B_true_mocks_mean,
            Qxx_opt_mocks_CRMS, Qxy_opt_mocks_CRMS, Qxz_opt_mocks_CRMS,
            Qyy_opt_mocks_CRMS, Qyz_opt_mocks_CRMS, Qzz_opt_mocks_CRMS,
            Qxx_true_mocks_CRMS, Qxy_true_mocks_CRMS, Qxz_true_mocks_CRMS,
            Qyy_true_mocks_CRMS, Qyz_true_mocks_CRMS, Qzz_true_mocks_CRMS,
            Bx_opt_mocks_mean, By_opt_mocks_mean, Bz_opt_mocks_mean, 
            Bx_true_mocks_mean, By_true_mocks_mean, Bz_true_mocks_mean,
            Qxx_opt_mocks_mean, Qxy_opt_mocks_mean, Qxz_opt_mocks_mean,
            Qyy_opt_mocks_mean, Qyz_opt_mocks_mean, Qzz_opt_mocks_mean,
            Qxx_true_mocks_mean, Qxy_true_mocks_mean, Qxz_true_mocks_mean,
            Qyy_true_mocks_mean, Qyz_true_mocks_mean, Qzz_true_mocks_mean)

# Utilisation (inchangée)
Input_v5 = '/renoir/fromenti/Documents/codes_Bulk_flow/results_V5_data/'

(Bx, By, Bz, B_ampl, err_Bx, err_By, err_Bz, err_B_ampl,
            Qxx, Qxy, Qxz, Qyy, Qyz, Qzz,
            err_Qxx, err_Qxy, err_Qxz, err_Qyy, err_Qyz, err_Qzz,
            cov, SigV,
            Bx_opt_mocks_err, By_opt_mocks_err, Bz_opt_mocks_err,
            Bx_true_mocks_err, By_true_mocks_err, Bz_true_mocks_err,
            B_opt_mocks_err, B_true_mocks_err, B_opt_mocks_mean, B_true_mocks_mean,
            Qxx_opt_mocks_err, Qxy_opt_mocks_err, Qxz_opt_mocks_err,
            Qyy_opt_mocks_err, Qyz_opt_mocks_err, Qzz_opt_mocks_err,
            Qxx_true_mocks_err, Qxy_true_mocks_err, Qxz_true_mocks_err,
            Qyy_true_mocks_err, Qyz_true_mocks_err, Qzz_true_mocks_err,
            Bx_opt_mocks_mean, By_opt_mocks_mean, Bz_opt_mocks_mean, 
            Bx_true_mocks_mean, By_true_mocks_mean, Bz_true_mocks_mean,
            Qxx_opt_mocks_mean, Qxy_opt_mocks_mean, Qxz_opt_mocks_mean,
            Qyy_opt_mocks_mean, Qyz_opt_mocks_mean, Qzz_opt_mocks_mean,
            Qxx_true_mocks_mean, Qxy_true_mocks_mean, Qxz_true_mocks_mean,
            Qyy_true_mocks_mean, Qyz_true_mocks_mean, Qzz_true_mocks_mean) = load_data_all(
    Input_v5 + 'all_moments_data_l_b_etaMLE.txt',
    Input_v5 + 'all_moments_all_mocks_l_b_etaMLE.txt')


## Bulk flow direction

In [36]:
def bulk_flow_direction(Bx, By, Bz, err_Bx, err_By, err_Bz):
    
    # Amplitude et son erreur
    B_ampl = np.sqrt(Bx**2 + By**2 + Bz**2)
    err_B_ampl = (np.abs(Bx*err_Bx) + np.abs(By*err_By) + np.abs(Bz*err_Bz)) / B_ampl
    
    # Conversion en coordonnées galactiques
    cart = CartesianRepresentation(Bx, By, Bz, unit=u.km/u.s)
    sph = SkyCoord(cart, frame='galactic').spherical
    l, b = sph.lon.deg, sph.lat.deg
    
    # Erreurs sur l et b
    R_perp = np.sqrt(Bx**2 + By**2)
    err_l_deg = np.degrees(np.sqrt(((-By/R_perp**2)*err_Bx)**2 + ((Bx/R_perp**2)*err_By)**2)) if R_perp > 0 else 0
    err_b_deg = np.degrees(np.sqrt(((-Bx*Bz/(B_ampl**2*R_perp))*err_Bx)**2 + ((-By*Bz/(B_ampl**2*R_perp))*err_By)**2 + ((R_perp/B_ampl**2)*err_Bz)**2)) if (B_ampl > 0 and R_perp > 0) else 0
    
    return l, err_l_deg, b, err_b_deg

l,err_l, b, err_b = bulk_flow_direction(Bx, By, Bz, err_Bx, err_By, err_Bz)

print("Bulk Flow direction (l, b) = ({:.2f} ± {:.2f} deg, {:.2f} ± {:.2f} deg)".format(l, err_l, b, err_b))

Bulk Flow direction (l, b) = (291.67 ± 4.79 deg, 2.48 ± 3.51 deg)


## calculation of startistical test chi2 and associated p-value

In [23]:
# chi2 of all the bulk flow and the shear moments formula from paper of Fei Qin :

def chi2_lu(moment_measure, cov_prediction, Cov_mesure , M_prediction):
    
    # here the matrix moment_measured is the matrix of the data, err_prediction is the theoretical Cosmic root square matrix and
    # Cov is the covariance matrix of the MCMC chain data and M_theorique is the prediction of Lambda CDM or mocks
        
    U = moment_measure - M_prediction
    U = np.ravel(U)  # force U to be 1D vector 
    
    Tot_cov = cov_prediction + Cov_mesure
    Tot_cov_inv = np.linalg.inv(Tot_cov)

    chi2 = U @ Tot_cov_inv @ U
    
    return chi2

In [31]:
# chi2 and p-valuecalculation for the Bulk Flow compared to the Lambda CDM theory :
B_theorie = np.array([0,0,0])
cov_B_theorie = R_mean[:3, :3]

B_measured = np.array([Bx, By, Bz])
Cov_B = cov[:3, :3]

chi2_B_theorie = chi2_lu(B_measured, cov_B_theorie, Cov_B, B_theorie)
print(f"χ² pour le Bulk Flow = {chi2_B_theorie:.4f}")
chi2_norm_B_theorie = chi2_B_theorie / 3
print(f"χ² normalisé pour le Bulk Flow (χ²/3) = {chi2_norm_B_theorie:.4f}")
p_value_B_theorie = chi2.sf(chi2_B_theorie, 3)
print(f"p-value pour le Bulk Flow = {p_value_B_theorie:.7f}")


χ² pour le Bulk Flow = 14.9465
χ² normalisé pour le Bulk Flow (χ²/3) = 4.9822
p-value pour le Bulk Flow = 0.0018629


In [ ]:
# chi2 and p-value calculation for all the moments compared to the Lambda CDM theory :
all_theorie = np.array([0,0,0,0,0,0,0,0])
cov_all_theorie = R_mean[:8, :8]

all_measured = np.array([Bx, By, Bz, Qxx, Qxy, Qxz, Qyy, Qyz])
Cov_all = cov[:8, :8]

chi2_all_theorie = chi2_lu(all_measured, cov_all_theorie, Cov_all, all_theorie)
print(f"χ² pour tout les moments = {chi2_all_theorie:.4f}")
chi2_norm_all_theorie = chi2_all_theorie / 8
print(f"χ² normalisé pour tout les moments (χ²/8) = {chi2_norm_all_theorie:.4f}")
p_value_all_theorie = chi2.sf(chi2_all_theorie, 8)
print(f"p-value pour tout les moments = {p_value_all_theorie:.7f}")

χ² pour tout les moments = 23.8944
χ² normalisé pour tout les moments (χ²/8) = 2.9868
p-value pour tout les moments = 0.0023871


In [ ]:
# chi2 and p-value calculation for the Bulk Flow compared to the mocks estimated:

B_mocks_estimated = np.array([Bx_opt_mocks_mean, By_opt_mocks_mean, Bz_opt_mocks_mean])

# we neglect the correlations between the components because I don't have the covariance matrix of the mocks
cov_B_mocks_estimated = np.diag([Bx_opt_mocks_err**2, By_opt_mocks_err**2, Bz_opt_mocks_err**2]) # no diagonal elements 

B_measured = np.array([Bx, By, Bz])
Cov_B = cov[:3, :3]

chi2_B_mocks_estimated = chi2_lu(B_measured, cov_B_mocks_estimated, Cov_B, B_mocks_estimated)
print(f"χ² pour le Bulk Flow = {chi2_B_mocks_estimated:.4f}")
chi2_norm_B_mocks_estimated = chi2_B_mocks_estimated / 3
print(f"χ² normalisé pour le Bulk Flow (χ²/3) = {chi2_norm_B_mocks_estimated:.4f}")
p_value_B_mocks_estimated = chi2.sf(chi2_B_mocks_estimated, 3)
print(f"p-value pour le Bulk Flow = {p_value_B_mocks_estimated:.7f}")


χ² pour le Bulk Flow = 17.0275
χ² normalisé pour le Bulk Flow (χ²/3) = 5.6758
p-value pour le Bulk Flow = 0.0006976


In [ ]:
# chi2 and p-value calculation for the Bulk Flow compared to the mocks vrais:

B_mocks_true = np.array([B_true_mocks_mean, By_true_mocks_mean, Bz_true_mocks_mean])

# we neglect the correlations between the components because I don't have the covariance matrix of the mocks
cov_B_mocks_true = np.diag([Bx_true_mocks_err**2, By_true_mocks_err**2, Bz_true_mocks_err**2])

B_measured = np.array([Bx, By, Bz])
Cov_B = cov[:3, :3]

chi2_B_mocks_true = chi2_lu(B_measured, cov_B_mocks_true, Cov_B, B_mocks_true)
print(f"χ² pour le Bulk Flow = {chi2_B_mocks_true:.4f}")
chi2_norm_B_mocks_true = chi2_B_mocks_true / 3
print(f"χ² normalisé pour le Bulk Flow (χ²/3) = {chi2_norm_B_mocks_true:.4f}")
p_value_B_mocks_true = chi2.sf(chi2_B_mocks_true, 3)
print(f"p-value pour le Bulk Flow = {p_value_B_mocks_true:.7f}")


χ² pour le Bulk Flow = 13.7278
χ² normalisé pour le Bulk Flow (χ²/3) = 4.5759
p-value pour le Bulk Flow = 0.0033001


In [ ]:
#n chi2 and p-value calculation for all the moments compared to the mocks estimated:

all_mocks_estimated = np.array([Bx_opt_mocks_mean, By_opt_mocks_mean, Bz_opt_mocks_mean, Qxx_opt_mocks_mean, Qxy_opt_mocks_mean, Qxz_opt_mocks_mean, Qyy_opt_mocks_mean, Qyz_opt_mocks_mean])

# we neglect the correlations between the components because I don't have the covariance matrix of the mocks
cov_all_mocks_estimated = np.diag([Bx_opt_mocks_err**2, By_opt_mocks_err**2, Bz_opt_mocks_err**2, Qxx_opt_mocks_err**2, Qxy_opt_mocks_err**2, Qxz_opt_mocks_err**2, Qyy_opt_mocks_err**2, Qyz_opt_mocks_err**2])

all_measured = np.array([Bx, By, Bz, Qxx, Qxy, Qxz, Qyy, Qyz])
Cov_B = cov[:8, :8]

chi2_all_mocks_estimated = chi2_lu(all_measured, cov_all_mocks_estimated, Cov_B, all_mocks_estimated)
print(f"χ² pour le Bulk Flow = {chi2_all_mocks_estimated:.4f}")
chi2_norm_all_mocks_estimated = chi2_all_mocks_estimated / 8
print(f"χ² normalisé pour le Bulk Flow (χ²/8) = {chi2_norm_all_mocks_estimated:.4f}")
p_value_all_mocks_estimated = chi2.sf(chi2_all_mocks_estimated, 8)
print(f"p-value pour le Bulk Flow = {p_value_all_mocks_estimated:.7f}")

χ² pour le Bulk Flow = 19.9494
χ² normalisé pour le Bulk Flow (χ²/8) = 2.4937
p-value pour le Bulk Flow = 0.0105292


In [33]:
# chi2 and p-value calculation for all the moments compared to the mocks vrais:

all_mocks_true = np.array([Bx_opt_mocks_mean, By_opt_mocks_mean, Bz_opt_mocks_mean, Qxx_true_mocks_mean, Qxy_true_mocks_mean, Qxz_true_mocks_mean, Qyy_true_mocks_mean, Qyz_true_mocks_mean])

cov_all_mocks_true = np.diag([Bx_opt_mocks_err**2, By_opt_mocks_err**2, Bz_opt_mocks_err**2, Qxx_true_mocks_err**2, Qxy_true_mocks_err**2, Qxz_true_mocks_err**2, Qyy_true_mocks_err**2, Qyz_true_mocks_err**2])

all_measured = np.array([Bx, By, Bz, Qxx, Qxy, Qxz, Qyy, Qyz])
Cov_B = cov[:8, :8]

chi2_all_mocks_true = chi2_lu(all_measured, cov_all_mocks_true, Cov_B, all_mocks_true)
print(f"χ² pour le Bulk Flow = {chi2_all_mocks_true:.4f}")
chi2_norm_all_mocks_true = chi2_all_mocks_true / 8
print(f"χ² normalisé pour le Bulk Flow (χ²/8) = {chi2_norm_all_mocks_true:.4f}")
p_value_all_mocks_true = chi2.sf(chi2_all_mocks_true, 8)
print(f"p-value pour le Bulk Flow = {p_value_all_mocks_true:.7f}")

χ² pour le Bulk Flow = 21.1907
χ² normalisé pour le Bulk Flow (χ²/8) = 2.6488
p-value pour le Bulk Flow = 0.0066578


## Table of results 

In [37]:
print("Table - Bulk Flow and shear moment Measured from DESI DR1 Data 3000 galaxies in galactics coordinates")
print("="*115)
print("Component           Measured (ηMLE)     CRMS (ΛCDM)          True mocks(ηMLE)          Estimated mocks(ηMLE)")
print("-"*115)
print(f"Bx  (km/s)           {Bx:6.2f} ± {err_Bx:5.2f}      0 ± {Bx_LCDM:6.2f}          {Bx_true_mocks_mean:6.2f} ±{Bx_true_mocks_err:6.2f}             {Bx_opt_mocks_mean:6.2f}  ±{Bx_opt_mocks_err:6.2f} ")
print(f"By  (km/s)          {By:6.2f} ± {err_By:5.2f}      0 ± {By_LCDM:6.2f}          {By_true_mocks_mean:6.2f} ± {By_true_mocks_err:6.2f}            {By_opt_mocks_mean:6.2f}  ±{By_opt_mocks_err:6.2f} ")
print(f"Bz  (km/s)           {Bz:6.2f} ± {err_Bz:5.2f}      0 ± {Bz_LCDM:6.2f}          {Bz_true_mocks_mean:6.2f} ±{Bz_true_mocks_err:6.2f}             {Bz_opt_mocks_mean:6.2f}  ±{Bz_opt_mocks_err:6.2f} ")
print(f"|B| (km/s)           {B_ampl:6.2f} ± {err_B_ampl:5.2f}      0 ± {B_LCDM:6.2f}          {B_true_mocks_mean:6.2f} ± {B_true_mocks_err:6.2f}            {B_opt_mocks_mean:6.2f}  ± {B_opt_mocks_err:6.2f} ")
print(f"Qyy (h*km/s/Mpc)     {Qyy:6.3f} ± {err_Qyy:5.3f}      0 ±{Qyy_LCDM:6.3f}           {Qyy_true_mocks_mean:6.3f} ±{Qyy_true_mocks_err:6.3f}             {Qyy_opt_mocks_mean:6.3f}  ±{Qyy_opt_mocks_err:6.3f} ")
print(f"Qyz (h*km/s/Mpc)     {Qyz:6.3f} ± {err_Qyz:5.3f}      0 ±{Qyz_LCDM:6.3f}           {Qyz_true_mocks_mean:6.3f} ±{Qyz_true_mocks_err:6.3f}             {Qyz_opt_mocks_mean:6.3f}  ±{Qyz_opt_mocks_err:6.3f}")
print(f"Qzz (h*km/s/Mpc)     {Qzz:6.3f} ± {err_Qzz:5.3f}      0 ±{Qzz_LCDM:6.3f}           {Qzz_true_mocks_mean:6.3f} ±{Qzz_true_mocks_err:6.3f}             {Qzz_opt_mocks_mean:6.3f}  ±{Qzz_opt_mocks_err:6.3f} ")
print("")
print("-"*115)
print("")
print(f"Chi2  Bulk Flow                           {chi2_B_theorie:6.3f}                  {chi2_B_mocks_true:6.3f}                      {chi2_B_mocks_estimated:6.3f}           ddof : 3                                                         " )   
print(f"Chi2 normalized   Bulk Flow               {chi2_norm_B_theorie:6.3f}                  {chi2_norm_B_mocks_true:6.3f}                      {chi2_norm_B_mocks_estimated:6.3f} ")
print(f"P-value  Bulk Flow                         {p_value_B_theorie:.7f}               {p_value_B_mocks_true:.7f}                   {p_value_B_mocks_estimated:.7f}  "  )  
print("")
print(f"Direction of the Bulk flow        l = {l:6.2f}° ±{err_l:6.2f}°             b = {b:6.2f}° ±{err_b:6.2f}°             ")
print("")
print("-"*115)
print("")
print(f"Chi2  all moments                         {chi2_all_theorie:6.3f}                  {chi2_all_mocks_true:6.3f}                      {chi2_all_mocks_estimated:6.3f}           ddof : 8                                                           " )   
print(f"Chi2 normalized   all moments             {chi2_norm_all_theorie:6.3f}                  {chi2_norm_all_mocks_true:6.3f}                      {chi2_norm_all_mocks_estimated:6.3f} ")
print(f"P-value  all moments                       {p_value_all_theorie:.7f}               {p_value_all_mocks_true:.7f}                   {p_value_all_mocks_estimated:.7f}  "  )

Table - Bulk Flow and shear moment Measured from DESI DR1 Data 3000 galaxies in galactics coordinates
Component           Measured (ηMLE)     CRMS (ΛCDM)          True mocks(ηMLE)          Estimated mocks(ηMLE)
-------------------------------------------------------------------------------------------------------------------
Bx  (km/s)           169.10 ± 38.38      0 ± 135.53           27.16 ± 93.58              28.72  ± 75.62 
By  (km/s)          -425.50 ± 37.46      0 ± 159.45          -33.79 ± 106.03            -39.50  ± 95.64 
Bz  (km/s)            19.85 ± 28.07      0 ± 126.78          -22.80 ± 81.10             -24.41  ± 70.81 
|B| (km/s)           458.29 ± 50.15      0 ± 244.68           48.98 ± 163.02             54.59  ± 141.00 
Qyy (h*km/s/Mpc)     -0.008 ± 0.307      0 ± 0.608            0.077 ± 0.396              0.047  ± 0.537 
Qyz (h*km/s/Mpc)      0.610 ± 0.183      0 ± 0.488            0.126 ± 0.355              0.114  ± 0.422
Qzz (h*km/s/Mpc)      0.899 ± 0.431      0 